<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [2]:
%pip install torchinfo -qq
%pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
%pip install starfile -qq
%pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     / 10.7 MB 16.2 MB/s 0:00:01
  Preparing metadata (setup.py) ... done


> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [3]:
%pip install pycuda==2024.1
%pip install "numpy<2.0"
%pip install mrcfile -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 10.4 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2024.1-cp312-cp312-linux_x86_64.whl size=658547 sha256=2961836722e8360b4bd9540aef51043b723e3aa65c984d06009f44ec245b1973
  Stored in directory: /root/.cache/pip/wheels/16/8d/d0/cf29b67ddc32f77bc3344b6b2b38dd9fe8595c726c33eabc03
Successfully built pycuda
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 116.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.9 MB/s eta 0:00:00


## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [1]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C-CSN-D_float-u_output/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C-CSN-D_float-u_output/results/10017_test/unet_eb5_dice_CDCRF" # @param {type:"string"}

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
IMAGE_DIR = "/content/image_dir"

In [8]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

Cloning into 'CryoParticleSegment'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 270 (delta 141), reused 42 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 32.01 MiB | 16.73 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [3]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

> #### ⚠ Notice
>
> You need to restart the kernel after the compilation step. Additionally, if your GPU architecture differs, you may need to modify the file at `/content/CryoParticleSegment/Modeling/CRF_main/setup.py`. (We attempt to detect the GPU configuration automatically, so this step is usually not necessary.)
> Furthermore, depending on the number of classes and other parameters, you may encounter an error that requires adjusting the file at `/content/CryoParticleSegment/Modeling/CRF_main/src/PermutohedralFiltering/source/gpu/LatticeFilter.cu`:
>
> 1. Note the `pd` and `vd` values from the error message.
> 2. Add an `else if (pd == ? && vd == ?)` block.
> 3. Within that block, insert the following line, replacing `pd` and `vd` with the specific values:
>
>    ```cpp
>    latticeFilterGPU<pd, vd>(output_tensor, input_tensor, positions, num_super_pixels, backward);
>    ```
> 4. Recompile using `setup.py`.


In [10]:
#%git clone https://github.com/netw0rkf10w/CRF.git
%cd CryoParticleSegment/Modeling/CRF_main
!python setup.py clean --all
!rm -rf build/
!python setup.py build_ext --inplace --force
!python setup.py install

/content/CryoParticleSegment/Modeling/CRF_main
running clean
'build/lib.linux-x86_64-cpython-312' does not exist -- can't clean it
'build/bdist.linux-x86_64' does not exist -- can't clean it
'build/scripts-3.12' does not exist -- can't clean it
running build_ext
W1015 04:40:41.856000 3990 torch/utils/cpp_extension.py:615] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
W1015 04:40:41.926000 3990 torch/utils/cpp_extension.py:507] The detected CUDA version (12.5) has a minor version mismatch with the version that was used to compile PyTorch (12.6). Most likely this shouldn't be a problem.
W1015 04:40:41.927000 3990 torch/utils/cpp_extension.py:517] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.5
building 'Permutohedral' extension
creating build/temp.linux-x86_64-cpython-312/src/PermutohedralFiltering/source/cpu
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEB

In [4]:
%cd /content/

/content


### ✅ Packages Handling

In [5]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [6]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [7]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [9]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = True # @param {type:"boolean"}


In [10]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [11]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [12]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [13]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
train_dataset =  MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset , batch_size=None, shuffle=False, pin_memory=True)

val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True)

full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True)

In [14]:
shape = None
for i1, i2, i3, i4 in val_loader: #test loader and reconstruct
    shape = i4.shape
    break

In [15]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [16]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:28<00:00,  2.98it/s]


In [17]:
processed_micrographs.shape

(84, 3840, 3840)

In [18]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [19]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [20]:
del model
torch.cuda.empty_cache()

In [21]:
model = model_post

In [22]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/C-CSN-D_float-u_output/results/10017_test/unet_eb5_dice_CDCRF'

In [23]:
import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()
        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

Output hidden; open in https://colab.research.google.com to view.

In [24]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [25]:
!cp {RESULT_DIR}/label_images.npy .

In [26]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [27]:
label_images.shape

(84, 3840, 3840)

In [28]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [29]:
!cp {RESULT_DIR}/best_config.txt .

In [30]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

### Actual seg

In [31]:
radius = RADIUS

In [32]:
from skimage import filters, segmentation, morphology
import skimage as ski
from skimage import img_as_float
from center_finding import normalize, min_rect_circle, eliminate_near
import cv2

In [33]:
from skimage import img_as_float
from skimage.filters import threshold_local

cv_list = []
for img in label_images:
    output_image = img_as_float(img)
    block_size = int(round(radius * 1.6)) | 1  # Ensure it's an odd number
    local_thresh = filters.threshold_local(output_image, block_size, method='gaussian', offset=0)
    image_seg = output_image > local_thresh
    thresh1 = ski.util.img_as_ubyte(image_seg)

    contr_min = radius*cv_config[1]
    kernel_size = int(radius / cv_config[0])  # Example ratio
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    thresh1 = cv2.erode(thresh1, kernel, iterations=1)

    contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_array = [c for c in contours]

    # Filter out small/large region and find bounding box with center
    c_ = np.array([cv2.contourArea(contour) for contour in contours])
    aa = (c_>contr_min) & (c_<500000)
    aa = aa.tolist()
    c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
    c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
    c_list = [x for x in c_list if x is not None]
    cv_list.append(c_list)
    print(len(c_full_list), len(c_list))

549 472
542 471
481 373
394 386
404 380
490 416
335 332
458 411
534 450
584 503
522 459
518 474
537 467
478 449
573 484
539 474
519 458
532 472
498 454
561 494
551 462
533 447
530 463
533 416
482 385
488 394
505 399
544 436
524 423
561 507
536 462
492 383
535 448
539 457
526 447
532 439
514 434
544 447
544 496
526 441
429 385
543 450
562 501
388 375
285 283
399 359
325 320
524 506
569 489
497 421
522 448
522 444
541 471
563 551
563 509
492 448
517 436
585 527
446 349
503 421
446 438
479 425
492 453
478 446
455 424
460 431
406 385
458 402
393 341
431 377
455 402
162 162
453 400
475 435
424 392
488 426
454 437
458 404
465 432
359 356
433 370
455 398
407 351
430 375


In [34]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    #if idx == 6:
    #    break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in cv_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [35]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [36]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in cv_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [37]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1508,3951,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1235,3949,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,248,3948,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,753,3944,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,3712,3944,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
35815,1982,172,00370@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35816,2355,187,00371@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35817,316,148,00372@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
35818,2855,154,00373@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [38]:
import starfile
starfile.write(df, f'{RESULT_DIR}/cv_particles.star')

### TrackPy

In [39]:
cv_list

[[(1380, 3823),
  (1107, 3821),
  (120, 3820),
  (625, 3816),
  (3584, 3816),
  (2410, 3815),
  (3041, 3808),
  (1187, 3803),
  (2823, 3783),
  (2695, 3789),
  (3684, 3767),
  (1565, 3780),
  (2950, 3760),
  (3822, 3734),
  (3200, 3748),
  (975, 3732),
  (1295, 3730),
  (467, 3684),
  (1789, 3680),
  (365, 3669),
  (1136, 3652),
  (3713, 3661),
  (1530, 3657),
  (3257, 3631),
  (2767, 3658),
  (2985, 3628),
  (1439, 3667),
  (3053, 3647),
  (3809, 3622),
  (3689, 3570),
  (997, 3591),
  (755, 3564),
  (3089, 3557),
  (3226, 3548),
  (1127, 3538),
  (1908, 3538),
  (3808, 3527),
  (39, 3530),
  (2016, 3546),
  (874, 3502),
  (674, 3518),
  (2363, 3491),
  (3000, 3493),
  (137, 3485),
  (1577, 3475),
  (3476, 3496),
  (2777, 3458),
  (601, 3470),
  (1774, 3463),
  (3783, 3458),
  (764, 3428),
  (1480, 3437),
  (2267, 3423),
  (999, 3411),
  (16, 3408),
  (2930, 3403),
  (2640, 3413),
  (513, 3409),
  (3148, 3374),
  (2807, 3360),
  (882, 3378),
  (672, 3350),
  (210, 3358),
  (1597, 3340

In [40]:
tp_config

(0.25, 0.4)

In [41]:
import trackpy as tp

In [42]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = round(TrackParticleSize*tp_config[1])
scale = tp_config[0]  # Scale factor (0.5 means reducing the size to half)

In [43]:
tp_list = []
for img in label_images:
    output_image = img
    small_image = cv2.resize(output_image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    # Adjust parameters based on the scale
    small_sep = int(sep * scale)
    small_diameter = int(TrackParticleSize * scale)
    if small_diameter % 2 == 0:
        small_diameter += 1
    coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
    #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
    coorTrack['x'] *= (1/scale)
    coorTrack['y'] *= (1/scale)
    coords = coorTrack[['x', 'y']].values
    tp_list.append(coords)
    print(len(coords))

617
578
628
354
391
564
307
473
611
635
573
501
582
482
667
593
548
558
520
601
609
647
605
693
604
603
626
657
631
586
575
653
595
587
599
632
572
639
563
621
443
633
577
358
241
417
290
480
624
582
596
587
573
526
583
497
584
603
636
595
388
500
470
445
451
440
391
498
434
474
497
132
477
462
431
525
418
487
439
309
493
499
445
474


In [44]:
tp_list

[array([[1008.15981535,   59.07580883],
        [1531.18396358,  109.10529889],
        [1337.6066628 ,   95.79841915],
        ...,
        [2031.1929988 , 3746.56552955],
        [3200.10854964, 3744.10976213],
        [3682.91655054, 3765.02032954]]),
 array([[ 225.71250198,   78.946104  ],
        [1978.90775616,   41.3108959 ],
        [1779.18996387,   52.06089887],
        ...,
        [ 470.03747113, 3762.25454229],
        [2589.86142876, 3790.7481053 ],
        [2778.31194238, 3782.86582823]]),
 array([[3723.09324041,   41.55350042],
        [2100.00955285,   55.08145644],
        [2961.20820096,  100.94514339],
        ...,
        [ 799.10276498, 3740.53559472],
        [2746.17143929, 3780.34190342],
        [3689.3751791 , 3778.92205931]]),
 array([[1436.26279029,   71.98690414],
        [1254.99855659,   67.94219328],
        [1787.41822869,  149.19705944],
        [2385.19616491,  101.7071908 ],
        [3446.74953172,  172.8563598 ],
        [3219.74694561,  191.837322

In [45]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in tp_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [46]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in tp_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [47]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1136.159815,187.075809,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1659.183964,237.105299,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,1465.606663,223.798419,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,3141.530302,202.549092,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,3610.554923,199.886462,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
44079,1620.113750,3862.186700,00469@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
44080,1531.633428,3905.345010,00470@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
44081,3369.605005,3909.355045,00471@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
44082,3447.613768,3857.282884,00472@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [48]:
import starfile
starfile.write(df, f'{RESULT_DIR}/tp_particles.star')

### Nonmax

In [49]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [50]:
nms_config

(0.5, 0.6)

In [51]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more
pCan=nms_config[0] #the grid size smaller will generate more candidate
overlap=nms_config[1] #allow for overlap
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [52]:
nms_list = []
for idx, img in enumerate(label_images):
    # if idx == 6:
    #     break
    heatArr=pad_image(img)
    heatArr=reNorm(heatArr, nSep)
    heatArr=reLev(heatArr,level)
    gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

    # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
    gsize=psize*pCan
    heat=heatArr.astype(np.float32)
    gaus=gaus.astype(np.float32)
    score=np.zeros(heat.shape).astype(np.float32)
    [sizex,sizey]=heat.shape
    sizex = int(sizex)
    sizey = int(sizey)
    psize = int(psize)
    gsize = int(gsize)

    func = scoreGpu

    tx=16
    ty=16
    bx=(sizex-1)//tx+1
    by=(sizey-1)//ty+1
    print('get score gpu',tx, ty, bx, by, gsize, psize)


    func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
        block=(tx, ty, 1), grid=(int(bx), int(by)))

    # # Calculate necessary dimensions and convert all to int
    numx = int((sizex - 1) // gsize + 1)
    numy = int((sizey - 1) // gsize + 1)
    num = numx * numy
    tnum = num * 5

    # # Create the result array
    res = np.zeros(tnum).astype(np.float32)

    # # Block and grid dimensions
    bx = (numx - 1) // tx + 1
    by = (numy - 1) // ty + 1

    # Get the function from the module
    func = getMax

    # Call the function, ensure all parameters are the correct type
    func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
        block=(16, 16, 1), grid=(bx, by, 1))

    # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
    niter = np.int32(nIter)
    gsize = np.int32(gsize)
    sizex = np.int32(sizex)
    sizey = np.int32(sizey)
    numx = np.int32(numx)
    tnum = np.int32(num)

    print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
    func = getMax3
    # Ensuring the correct parameter order and type for the kernel invocation
    func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
            block=(16, 16, 1), grid=(bx, by, 1))

    canList=reshape(res,num)

    print('Number of Particles before:', len(canList))
    if(len(canList)>pnum):
        canList=canList[:pnum]


    canList=cleanCanList(canList,overlap,psize)
    #canList=reCan(canList,ratio)
    print('Number of Particles:', len(canList))
    nms_list.append([(r[1],r[0]) for r in canList])

get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3470
Number of Particles: 656
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3462
Number of Particles: 602
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3472
Number of Particles: 686
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3263
Number of Particles: 440
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3261
Number of Particles: 451
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3840 60 3600
convert
Number of Particles before: 3426
Number of Particles: 625
get score gpu 16 16 240 240 64 128
get Max3 16 16 4 4 100 other :  64 3840 3

In [53]:
import matplotlib
for idx, (test_image, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
    # else:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in nms_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [54]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in nms_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [55]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,2458.0,359.0,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,2301.6,3759.2,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,351.0,2891.8,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,1137.6,1788.8,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,2255.0,577.0,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
49037,2688.0,1535.0,00559@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
49038,3840.0,1791.0,00560@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
49039,2048.0,3328.0,00561@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
49040,2687.0,3712.0,00562@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [56]:
starfile.write(df, f'{RESULT_DIR}/nms_particles.star')